In [1]:
#!/usr/bin/env python3
# -*- coding: utf-8 -*-
"""
Fusiona los JSONs de encabezado, asunto y presidente en un solo archivo enriquecido,
ignorando registros con campos vacíos o nulos.
Incluye múltiples formatos de fecha: fecha_yyyymmdd, fecha_ddmmyyyy, fecha_larga, fecha_corta, hora
"""

import json
from pathlib import Path
import re
from datetime import datetime
import uuid

# ──────── Directorios ─────────────────────────────────────────────────────────
base_path = Path("/home/nahumfg/Projects/GithubProjects/tesismaestriauni-launcher/publicdata-yolo-ocr/notebooks_ocr/votacion/jsons")
dir_encabezado = base_path / "encabezado"
dir_asunto = base_path / "asunto"
dir_presidente = base_path / "presidente"

# ──────── Regex útiles ────────────────────────────────────────────────────────
re_sesion = re.compile(r"([^/\\]+)$")
re_pp = re.compile(r"pp(?P<start>\d{4})_(?P<end>\d{4})")
re_pa = re.compile(r"pa(?P<start>\d{4})_(?P<end>\d{4})")
re_leg = re.compile(r"leg(\d)")
re_page = re.compile(r"page_(\d+)")

# ──────── Constantes para fechas en español ──────────────────────────────────
MESES_ES    = ["enero","febrero","marzo","abril","mayo","junio",
               "julio","agosto","septiembre","octubre","noviembre","diciembre"]
MESES_ABBR  = ["ene","feb","mar","abr","may","jun",
               "jul","ago","sep","oct","nov","dic"]

# ──────── Indexar archivos por sesion ─────────────────────────────────────────────────────────
def extract_sesion(path_str: str) -> str:
    match = re_sesion.search(path_str)
    return match.group(1) if match else None

def load_jsons_by_sesion(dir_path: Path, key: str):
    mapping = {}
    for path in dir_path.glob("*.json"):
        with path.open(encoding="utf-8") as f:
            data = json.load(f)
            sesion = extract_sesion(data.get(key, ""))
            if sesion:
                mapping[sesion] = data
    return mapping

encabezados = load_jsons_by_sesion(dir_encabezado, "source_dir")
asuntos = load_jsons_by_sesion(dir_asunto, "source_path")
presidentes = load_jsons_by_sesion(dir_presidente, "source_path")

# ──────── Fusionar ────────────────────────────────────────────────────────────
records = []
omitidos = 0

for sesion, enc in encabezados.items():
    try:
        fecha = enc.get("fecha", "")
        utc5 = enc.get("utc_5", fecha)

        # ── Validación y procesamiento de la fecha ISO ──────────────────────
        try:
            dt = datetime.fromisoformat(fecha)
            fecha_human = dt.strftime("%d %b %Y %H:%M")
            
            # ── Nuevos formatos de fecha ────────────────────────────────────
            fecha_yyyymmdd = dt.strftime("%Y-%m-%d")
            fecha_ddmmyyyy = dt.strftime("%d/%m/%Y")
            fecha_larga = f"{dt.day:02d} de {MESES_ES[dt.month-1]} del {dt.year}"
            fecha_corta = f"{dt.day:02d} {MESES_ABBR[dt.month-1]} {dt.year}"
            hora = dt.strftime("%H:%M")
        except Exception:
            # Si no se puede parsear la fecha, omitir este registro
            omitidos += 1
            continue

        asunto = asuntos.get(sesion, {}).get("asunto", "")
        presidente = presidentes.get(sesion, {}).get("presidente", "")

        pp_match = re_pp.search(sesion)
        pa_match = re_pa.search(sesion)
        pc_start = int(pp_match.group("start")) if pp_match else None
        pc_end = int(pp_match.group("end")) if pp_match else None
        pa_start = int(pa_match.group("start")) if pa_match else None
        pa_end = int(pa_match.group("end")) if pa_match else None
        n_legislatura = int(re_leg.search(sesion).group(1)) if re_leg.search(sesion) else None
        page = int(re_page.search(sesion).group(1)) if re_page.search(sesion) else None

        record = {
            "id": str(uuid.uuid4()),
            "sesion": sesion,
            "fecha": fecha,
            "fecha_utc5": utc5,
            "fecha_yyyymmdd": fecha_yyyymmdd,
            "fecha_ddmmyyyy": fecha_ddmmyyyy,
            "fecha_larga": fecha_larga,
            "fecha_corta": fecha_corta,
            "hora": hora,
            "legislatura": enc.get("legislatura", ""),
            "tipo": enc.get("tipo", ""),
            "periodo_congreso_inicio": pc_start,
            "periodo_congreso_fin": pc_end,
            "periodo_congreso": f"{pc_start}-{pc_end}" if pc_start and pc_end else "",
            "periodo_anual_inicio": pa_start,
            "periodo_anual_fin": pa_end,
            "periodo_anual": f"{pa_start}-{pa_end}" if pa_start and pa_end else "",
            "n_legislatura": n_legislatura,
            "page": page,
            "asunto": asunto,
            "presidente": presidente,
            "url": f"http://localhost:8080/votacion/{sesion}.png"
        }

        if all(v not in [None, ""] for v in record.values()):
            records.append(record)
        else:
            omitidos += 1

    except Exception as e:
        print(f"⚠️ Error en {sesion}: {e}")
        omitidos += 1

# ──────── Guardar resultado ───────────────────────────────────────────────────
output_path = Path("../data/voting_docs_enriched.json")
records.sort(key=lambda r: r["fecha_yyyymmdd"])

with output_path.open("w", encoding="utf-8") as f:
    json.dump(records, f, ensure_ascii=False, indent=2)

print(f"✅ Generado {output_path} ({len(records)} registros válidos)")
print(f"❌ Registros omitidos por campos vacíos o nulos: {omitidos}")


✅ Generado ../data/attendance_docs_votacion_enriched.json (7591 registros válidos)
❌ Registros omitidos por campos vacíos o nulos: 943
